Imports, load data from HF Block 1

In [ ]:
!pip -q install datasets pandas scikit-learn
from datasets import load_dataset
import pandas as pd, hashlib

ds = load_dataset("lmarena-ai/arena-human-preference-140k", split="train")
ds = ds.select_columns(["model_a","model_b","conversation_a","conversation_b","language","is_code","category_tag"])


Get a count of what models the set contains.  Block 2

In [ ]:
import pandas as pd

def family(name):
    m = name.lower()
    if "claude" in m: return "Claude"
    if "gemini" in m or "gemma" in m: return "Gemini"
    if "gpt" in m or m.startswith(("o1","o3","o4")): return "GPT"
    if "llama" in m: return "Llama"
    if "qwen" in m or "qwq" in m: return "Qwen"
    if "deepseek" in m: return "DeepSeek"
    if "grok" in m: return "Grok"
    if "mistral" in m or "mixtral" in m: return "Mistral"
    return "Other"

d = ds.select_columns(["model_a", "model_b"]).to_pandas()
models = pd.concat([d["model_a"], d["model_b"]], ignore_index=True)

counts = models.value_counts().rename_axis("model").reset_index(name="rows")
counts["family"] = counts["model"].map(family)

pd.set_option("display.max_rows", None)
print(counts)
print()
print(counts.groupby("family")["rows"].sum().sort_values(ascending=False))

Simplify problem to classifying each observation as GPT, Claude or Gemini Block 3

In [ ]:
import pandas as pd

keep = ["Claude", "Gemini", "GPT"]

d = ds.select_columns(["model_a", "model_b", "conversation_a", "conversation_b"]).to_pandas()

# each battle holds two model responses split them into one row per response
a = d[["model_a", "conversation_a"]].rename(columns={"model_a": "model", "conversation_a": "conversation"})
b = d[["model_b", "conversation_b"]].rename(columns={"model_b": "model", "conversation_b": "conversation"})
df = pd.concat([a, b], ignore_index=True)

df["family"] = df["model"].map(family)
df = df[df["family"].isin(keep)].reset_index(drop=True)

print(len(df))
print(df["family"].value_counts())

Extract rows so its LLM family, LLM input, LLM output Block 4

In [ ]:
def msg_text(msg):
    c = msg["content"]
    if isinstance(c, str):
        return c.strip()
    return "\n".join(p.get("text") or "" for p in c if p.get("type") == "text").strip()

def extract(conv):
    # single turn only exactly one user message then one LLM response
    if len(conv) != 2 or conv[0]["role"] != "user" or conv[1]["role"] != "assistant":
        return None, None
    return msg_text(conv[0]), msg_text(conv[1])

pairs = df["conversation"].map(extract)
df["llm_input"] = pairs.map(lambda p: p[0])
df["llm_output"] = pairs.map(lambda p: p[1])

before = len(df)
df = df.dropna(subset=["llm_input", "llm_output"])
df = df[(df["llm_input"] != "") & (df["llm_output"] != "")]
df = df.drop(columns=["conversation"]).reset_index(drop=True)

print(df["family"].value_counts())
df[["family", "model", "llm_input", "llm_output"]].head()

Filter to english only Block 5

In [ ]:
import pandas as pd

keep = ["Claude", "Gemini", "GPT"]

d = ds.select_columns(["model_a", "model_b", "conversation_a", "conversation_b", "language"]).to_pandas()

a = d[["model_a", "conversation_a", "language"]].rename(columns={"model_a": "model", "conversation_a": "conversation"})
b = d[["model_b", "conversation_b", "language"]].rename(columns={"model_b": "model", "conversation_b": "conversation"})
df = pd.concat([a, b], ignore_index=True)

# 3 families
df["family"] = df["model"].map(family)
df = df[df["family"].isin(keep)]

# single turn text extraction
pairs = df["conversation"].map(extract)
df["llm_input"] = pairs.map(lambda p: p[0])
df["llm_output"] = pairs.map(lambda p: p[1])
df = df.dropna(subset=["llm_input", "llm_output"])
df = df[(df["llm_input"] != "") & (df["llm_output"] != "")]
df = df.drop(columns=["conversation"])

print("before English filter:", len(df))
print(df["language"].value_counts().head(10))

# english only
df = df[df["language"] == "en"].reset_index(drop=True)

print("\nafter English filter:", len(df))
print(df["family"].value_counts())

Block 6. get counts of duplicates, repeated prompts reponse length sizes

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

# duplicates
n = len(df)
dup_full = df.duplicated(subset=["model", "llm_input", "llm_output"]).sum()
dup_out  = df.duplicated(subset=["family", "llm_input", "llm_output"]).sum()
print(" Dupes ")
print(f"rows: {n}")
print(f"exact duplicates (model + input + output): {dup_full} ({dup_full/n:.1%})")
print(f"duplicates within a family (family + input + output): {dup_out} ({dup_out/n:.1%})")
print()

#  repeated prompts
prompt_counts = df["llm_input"].value_counts()
uniq_prompts = len(prompt_counts)
repeated = prompt_counts[prompt_counts > 1]
print("Repeated prompts ")
print(f"unique prompts: {uniq_prompts}")
print(f"prompts that appear more than once: {len(repeated)} "
      f"(covering {repeated.sum()} rows, {repeated.sum()/n:.1%} of data)")


fam_per_prompt = df.groupby("llm_input")["family"].nunique()
multi = fam_per_prompt[fam_per_prompt > 1]
rows_multi = df["llm_input"].isin(multi.index).sum()
print(f"\nprompts answered by more than one family: {len(multi)} "
      f"(covering {rows_multi} rows, {rows_multi/n:.1%} of data)")
print()

#  response length per family
df_len = pd.DataFrame({
    "family": df["family"],
    "chars": df["llm_output"].str.len(),
    "words": df["llm_output"].str.split().str.len(),
})
print("response length per fam")
print(df_len.groupby("family")[["chars", "words"]].describe().T)

print("\nshort responses (< 20 chars) per family:")
print((df_len["chars"] < 20).groupby(df_len["family"]).sum())

print("\nrows per model within each family:")
print(df.groupby(["family", "model"]).size().sort_values(ascending=False)
        .rename("rows").reset_index())

Drop duplicates, cap each model within a family to 3k responses, Balance the families to 18k (GPT), then split and verify they are roughly even block 7

In [ ]:
import hashlib

SEED = 42
MODEL_CAP = 3000

# drop exact duplicates and responses under 20 characters
data = df.drop_duplicates(subset=["model", "llm_input", "llm_output"])
data = data[data["llm_output"].str.len() >= 20]
print("after dedupe + short-response drop:", len(data))

# capp each model at 3,000 rows
data = data.sample(frac=1, random_state=SEED)
data = data.groupby("model").head(MODEL_CAP)
print("\nafter per-model cap:")
print(data["family"].value_counts())

# balance families to the smallest one
n = data["family"].value_counts().min()
data = data.sample(frac=1, random_state=SEED).groupby("family").head(n)
print(f"\nafter balancing ({n} per family): {len(data)} rows")

# split 80/10/10 by prompt so no prompt appears in more than one split
def split_of(prompt):
    h = int(hashlib.md5(prompt.strip().encode()).hexdigest(), 16) % 100
    return "train" if h < 80 else ("val" if h < 90 else "test")

data["split"] = data["llm_input"].map(split_of)
data = data[["family", "model", "llm_input", "llm_output", "split"]].reset_index(drop=True)

# Checks
print("\nrows per split and family:")
print(pd.crosstab(data["split"], data["family"], margins=True))

leak = data.groupby("llm_input")["split"].nunique()
print("\nprompts appearing in more than one split:", (leak > 1).sum())

Prepare tokenizer and vocab Block 8

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader

MAX_LEN   = 512
MIN_FREQ  = 3
MAX_VOCAB = 50000
BATCH     = 64
PAD, UNK  = 0, 1

# keeps case punctuation and each newline as its own token spaces and tabs are dropped
TOKEN_RE = re.compile(r"\n|\w+|[^\w\s]")
def tokenize(text):
    return TOKEN_RE.findall(text)

# Vocabulary from the TRAIN split only
train = data[data["split"] == "train"]
counter = Counter()
for t in pd.concat([train["llm_input"], train["llm_output"]]):
    counter.update(tokenize(t))

itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(MAX_VOCAB - 2) if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
print("vocab size:", len(itos))

# Labels
classes  = sorted(data["family"].unique())
label2id = {c: i for i, c in enumerate(classes)}
print("classes:", label2id)

# cut to MAX_LEN, pad on the right, keep true lengths
def encode(text):
    ids = [stoi.get(t, UNK) for t in tokenize(text)[:MAX_LEN]]
    n = max(len(ids), 1)
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids, n

def build(split, column):
    d = data[data["split"] == split]
    enc = [encode(t) for t in d[column]]
    X = torch.tensor([e[0] for e in enc], dtype=torch.long)
    L = torch.tensor([e[1] for e in enc], dtype=torch.long)
    y = torch.tensor(d["family"].map(label2id).values, dtype=torch.long)
    return TensorDataset(X, L, y)

def make_loaders(column="llm_output"):
    return {s: DataLoader(build(s, column), batch_size=BATCH, shuffle=(s == "train"))
            for s in ["train", "val", "test"]}

loaders = make_loaders("llm_output")

# ---- Checks ----
Xv, Lv, yv = loaders["val"].dataset.tensors
print("\nval shape:", tuple(Xv.shape))
print(f"unknown-token rate (val): {(Xv == UNK).sum().item() / (Xv != PAD).sum().item():.2%}")
print(f"val responses truncated at {MAX_LEN} tokens: {(Lv == MAX_LEN).float().mean().item():.1%}")
print("train/val/test sizes:", {s: len(l.dataset) for s, l in loaders.items()})

xb, lb, yb = next(iter(loaders["train"]))
print("batch:", xb.shape, lb.shape, yb.shape)

block 9 check how many are cut off with 512 tokens, counts tokens for each response

In [ ]:
import pandas as pd

# token count per response before cut using the same tokenizer as the loaders
tok_len = data["llm_output"].map(lambda t: len(tokenize(t)))
report = pd.DataFrame({"family": data["family"], "split": data["split"], "tokens": tok_len})
report["truncated"] = report["tokens"] > MAX_LEN

print(" Share of responses truncated at 512 tokens ")
print((report.groupby("family")["truncated"].mean() * 100).round(1).rename("% truncated"))

print("\n Same, by split ")
print((report.pivot_table(index="family", columns="split", values="truncated", aggfunc="mean") * 100).round(1))

print("\nToken length per family ")
print(report.groupby("family")["tokens"].describe(percentiles=[.25, .5, .75, .9]).round(0))

print("\n Share that would be cut at other lengths ")
for L in [256, 512, 768, 1024, 1536]:
    share = (report["tokens"] > L).groupby(report["family"]).mean() * 100
    print(f"{L:>5}:", share.round(1).to_dict())

set up cnn and ltsm Block 10

In [ ]:
import copy, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

VOCAB = len(itos)
N_CLASSES = len(classes)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class CNN(nn.Module):
    def __init__(self, emb=128, filters=128, kernels=(3, 4, 5), dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(emb, filters, k) for k in kernels])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(filters * len(kernels), N_CLASSES)

    def forward(self, x, lengths):
        e = self.emb(x).transpose(1, 2)                      # (B, emb, T)
        pooled = [F.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(pooled, dim=1)))

class BiLSTM(nn.Module):
    def __init__(self, emb=128, hidden=128, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb, padding_idx=PAD)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, N_CLASSES)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(self.emb(x), lengths.cpu().clamp(min=1),
                                      batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)                         # h: (2, B, hidden)
        return self.fc(self.drop(torch.cat([h[0], h[1]], dim=1)))

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, ys = [], []
    for x, l, y in loader:
        out = model(x.to(device), l.to(device))
        preds.append(out.argmax(1).cpu()); ys.append(y)
    return torch.cat(preds).numpy(), torch.cat(ys).numpy()

def train_model(model, loaders, epochs=10, patience=2, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc, best_state, bad = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for x, l, y in loaders["train"]:
            x, l, y = x.to(device), l.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x, l), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(y)
        p, t = predict(model, loaders["val"])
        val_acc = accuracy_score(t, p)
        print(f"epoch {ep:2d} | train loss {total/len(loaders['train'].dataset):.4f} "
              f"| val acc {val_acc:.4f} | {time.time()-t0:.0f}s")
        if val_acc > best_acc:
            best_acc, best_state, bad = val_acc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                print("early stop"); break
    model.load_state_dict(best_state)
    return model, best_acc

def evaluate(model, loader, title):
    p, t = predict(model, loader)
    acc = accuracy_score(t, p)
    f1 = f1_score(t, p, average="macro")
    print(f"\n=== {title} | test acc {acc:.4f} | macro-F1 {f1:.4f} ===")
    print(classification_report(t, p, target_names=classes, digits=3))
    print("confusion matrix (rows = true, cols = predicted):")
    print(pd.DataFrame(confusion_matrix(t, p), index=classes, columns=classes))
    return {"acc": acc, "macro_f1": f1}

regular run

In [ ]:
SEED = 42
results = {}

for name, cls in [("CNN", CNN), ("BiLSTM", BiLSTM)]:
    print(f"\n {name} output only")
    set_seed(SEED)
    model, best_val = train_model(cls(), loaders)
    results[name] = evaluate(model, loaders["test"], f"{name} (output only)")
    results[name]["val_acc"] = best_val

print("\n RQ1 summary ")
print(pd.DataFrame(results).T[["val_acc", "acc", "macro_f1"]].round(4)
        .rename(columns={"acc": "test_acc"}))
print("chance:", round(1 / N_CLASSES, 4))

length controlled run block 12

In [ ]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

LC_LEN = 256

d = data.copy()
d["n_tokens"] = d["llm_output"].map(lambda t: len(tokenize(t)))

def balance_per_split(df, seed=42):
    """Equal rows per family within each split."""
    out = []
    for s, g in df.groupby("split"):
        n = g["family"].value_counts().min()
        out.append(g.groupby("family", group_keys=False).sample(n, random_state=seed))
    return pd.concat(out)

CONTROLLED = f"length-controlled (>= {LC_LEN} tokens, cut to {LC_LEN})"
REFERENCE  = f"reference (all responses, window {LC_LEN})"
conditions = {
    CONTROLLED: balance_per_split(d[d["n_tokens"] >= LC_LEN]),
    REFERENCE:  balance_per_split(d),
}

lc_results = {}
for cname, df_c in conditions.items():
    print(f"\n {cname} ")
    print(df_c.groupby(["split", "family"]).size().unstack())

    # build loaders inline: first LC_LEN tokens, padded, with true lengths
    ldrs = {}
    for s in ["train", "val", "test"]:
        g = df_c[df_c["split"] == s]
        X_rows, L_rows = [], []
        for text in g["llm_output"]:
            ids = [stoi.get(t, UNK) for t in tokenize(text)[:LC_LEN]]
            L_rows.append(max(len(ids), 1))
            X_rows.append(ids + [PAD] * (LC_LEN - len(ids)))
        X = torch.tensor(X_rows, dtype=torch.long)
        L = torch.tensor(L_rows, dtype=torch.long)
        y = torch.tensor(g["family"].map(label2id).values, dtype=torch.long)
        ldrs[s] = DataLoader(TensorDataset(X, L, y), batch_size=BATCH, shuffle=(s == "train"))

    if cname == CONTROLLED:
        for s in ["train", "val", "test"]:
            assert (ldrs[s].dataset.tensors[1] == LC_LEN).all()
        print("check passed all responses are exactly", LC_LEN, "tokens")

    for mname, cls in [("CNN", CNN), ("BiLSTM", BiLSTM)]:
        print(f"\n--- {mname} ---")
        set_seed(42)
        model, best_val = train_model(cls(), ldrs)
        r = evaluate(model, ldrs["test"], f"{mname} | {cname}")
        r["val_acc"] = best_val
        lc_results[(cname, mname)] = r

# summary
rows = []
for (cname, mname), r in lc_results.items():
    rows.append({"condition": cname, "model": mname, "val_acc": r["val_acc"],
                 "test_acc": r["acc"], "macro_f1": r["macro_f1"]})

print("\n Length-control summary ")
print(pd.DataFrame(rows).set_index(["condition", "model"]).round(4))
print("chance:", round(1 / N_CLASSES, 4))

Formating stripped run block 12

In [ ]:
import re
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D]")
RULE_RE  = re.compile(r"^[ \t:\-_]{3,}$", re.M)         # ---, ___, :---:, table separator rows
BULLET_RE = re.compile(r"^[ \t]*[-+•]\s+", re.M)         # "- item" bullets

def strip_formatting(t):
    t = re.sub(r"[*#|`]", "", t)             # bold/italic, headers, tables, code fences
    t = EMOJI_RE.sub("", t)                  # emoji
    t = re.sub(r"[—–]", " ", t)              # em and en dashes
    t = RULE_RE.sub("", t)                   # horizontal rules and table separators
    t = BULLET_RE.sub("", t)                 # hyphen / plus / dot bullets
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"^[ \t]*\d+[.)]\s+", "", t, flags=re.M)        # numbered lists
    t = re.sub(r"^>+\s?", "", t, flags=re.M)                    # blockquotes
    t = re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", t)              # links -> text
    t = re.sub(r"[→←↔⇒•]", " ", t)                              # arrows, inline bullets
    t = re.sub(r"[ \t]*\n[ \t]*", "\n", t)                      # strip each line
    return t.strip()

fmt_df = data.copy()
fmt_df["llm_output"] = fmt_df["llm_output"].map(strip_formatting)

# checks
joined = "\n".join(fmt_df["llm_output"])
print("remaining markdown/emoji/dash characters:",
      {c: joined.count(c) for c in ["*", "#", "|", "`", "—", "–"]},
      "| emoji:", len(EMOJI_RE.findall(joined)))
print("responses that became empty:", (fmt_df["llm_output"] == "").sum())
chars_before = data["llm_output"].str.len().sum(); chars_after = fmt_df["llm_output"].str.len().sum()
print(f"characters removed: {1 - chars_after/chars_before:.1%}")


# retrain on stripped text: build loaders inline (first MAX_LEN tokens, padded, with true lengths)
fmt_loaders = {}
for s in ["train", "val", "test"]:
    g = fmt_df[fmt_df["split"] == s]
    X_rows, L_rows = [], []
    for text in g["llm_output"]:
        ids = [stoi.get(t, UNK) for t in tokenize(text)[:MAX_LEN]]
        L_rows.append(max(len(ids), 1))
        X_rows.append(ids + [PAD] * (MAX_LEN - len(ids)))
    X = torch.tensor(X_rows, dtype=torch.long)
    L = torch.tensor(L_rows, dtype=torch.long)
    y = torch.tensor(g["family"].map(label2id).values, dtype=torch.long)
    fmt_loaders[s] = DataLoader(TensorDataset(X, L, y), batch_size=BATCH, shuffle=(s == "train"))

fmt_results = {}
for mname, cls in [("CNN", CNN), ("BiLSTM", BiLSTM)]:
    print(f"\n {mname} (formatting stripped)")
    set_seed(42)
    model, best_val = train_model(cls(), fmt_loaders)
    r = evaluate(model, fmt_loaders["test"], f"{mname} (formatting stripped)")
    r["val_acc"] = best_val
    fmt_results[mname] = r

# summary
rows = []
for m in ["CNN", "BiLSTM"]:
    rows.append({"model": m, "setting": "original", "test_acc": results[m]["acc"], "macro_f1": results[m]["macro_f1"]})
    rows.append({"model": m, "setting": "formatting stripped", "test_acc": fmt_results[m]["acc"], "macro_f1": fmt_results[m]["macro_f1"]})
summ = pd.DataFrame(rows).set_index(["model", "setting"])
print("\n=== Formatting ablation ===")
print(summ.round(4))

tf-idf baseline block 14

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def strip_formatting(t):
    t = re.sub(r"[*#|`]", "", t)             # bold/italic, headers, tables, code fences
    t = EMOJI_RE.sub("", t)                  # emoji
    t = re.sub(r"[—–]", " ", t)              # em and en dashes
    t = RULE_RE.sub("", t)                   # horizontal rules and table separators
    t = BULLET_RE.sub("", t)                 # hyphen / plus / dot bullets
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"^[ \t]*\d+[.)]\s+", "", t, flags=re.M)        # numbered lists
    t = re.sub(r"^>+\s?", "", t, flags=re.M)                    # blockquotes
    t = re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", t)              # links -> text
    t = re.sub(r"[→←↔⇒•]", " ", t)                              # arrows, inline bullets
    t = re.sub(r"[ \t]*\n[ \t]*", "\n", t)                      # strip each line
    return t.strip()

def bow_text(t):
    s = strip_formatting(t)
    return " ".join(tokenize(s)[:MAX_LEN])      # same first-512-token window the neural models saw

bow = data[["family", "split"]].copy()
bow["text"] = data["llm_output"].map(bow_text)
tr, va, te = [bow[bow["split"] == s] for s in ["train", "val", "test"]]

def run_bow(ngram, name):
    vec = TfidfVectorizer(lowercase=True, token_pattern=r"[a-zA-Z]+",
                          ngram_range=ngram, min_df=3, max_features=300000, sublinear_tf=True)
    Xtr = vec.fit_transform(tr["text"])
    Xva, Xte = vec.transform(va["text"]), vec.transform(te["text"])
    best = None
    for C in [1, 10]:
        clf = LogisticRegression(C=C, max_iter=1000).fit(Xtr, tr["family"])
        v = accuracy_score(va["family"], clf.predict(Xva))
        if best is None or v > best[0]:
            best = (v, C, clf)
    v, C, clf = best
    p = clf.predict(Xte)
    acc, f1 = accuracy_score(te["family"], p), f1_score(te["family"], p, average="macro")
    print(f"{name}: C={C} | val acc {v:.4f} | test acc {acc:.4f} | macro-F1 {f1:.4f}")
    return {"val_acc": v, "test_acc": acc, "macro_f1": f1}, vec, clf

r_uni, vec_uni, clf_uni = run_bow((1, 1), "bag of words (unigrams)")
r_bi,  vec_bi,  clf_bi  = run_bow((1, 2), "unigrams + bigrams")

# Most distinctive words per family (unigram model)
names = vec_uni.get_feature_names_out()
print("\nTop 15 words pushing toward each family ")
for i, fam in enumerate(clf_uni.classes_):
    top = clf_uni.coef_[i].argsort()[::-1][:15]
    print(f"{fam:7s}:", ", ".join(names[j] for j in top))

RQ4 analysis Block 15

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

WORD_RE  = re.compile(r"[A-Za-z']+")
SENT_RE  = re.compile(r"[.!?]+(?:\s|$)")
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
HEADER_RE = re.compile(r"^\s{0,3}#{1,6}\s", re.M)
BOLD_RE   = re.compile(r"\*\*[^*\n]+\*\*")
BULLET_RE = re.compile(r"^\s*[-*•]\s", re.M)
NUMBER_RE = re.compile(r"^\s*\d+[.)]\s", re.M)
TABLE_RE  = re.compile(r"^\s*\|", re.M)
HRULE_RE  = re.compile(r"^\s*(?:---+|\*\*\*+|___+)\s*$", re.M)

def mattr(words, window=100, cap=500):
    """Moving-average type-token ratio: lexical diversity that doesn't depend on length."""
    w = words[:cap]
    if len(w) < window:
        return np.nan
    c = Counter(w[:window]); total = len(c); n = 1
    for i in range(window, len(w)):
        out, inn = w[i - window], w[i]
        c[out] -= 1
        if c[out] == 0:
            del c[out]
        c[inn] += 1
        total += len(c); n += 1
    return total / (n * window)

def features(text):
    words = WORD_RE.findall(text)
    nw = max(len(words), 1)
    p = 100 / nw                                    # converts counts to "per 100 words"
    return {
        # length / structure
        "chars": len(text),
        "words": len(words),
        "lines": sum(1 for l in text.split("\n") if l.strip()),
        "paragraphs": sum(1 for x in re.split(r"\n\s*\n", text) if x.strip()),
        "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
        "words_per_sentence": nw / max(len(SENT_RE.findall(text)), 1),
        # lexical diversity
        "mattr": mattr([w.lower() for w in words]),
        # formatting (per 100 words)
        "headers": len(HEADER_RE.findall(text)) * p,
        "bold": len(BOLD_RE.findall(text)) * p,
        "bullets": len(BULLET_RE.findall(text)) * p,
        "numbered": len(NUMBER_RE.findall(text)) * p,
        "table_rows": len(TABLE_RE.findall(text)) * p,
        "hrules": len(HRULE_RE.findall(text)) * p,
        "code_blocks": (text.count("```") // 2) * p,
        # punctuation / tone (per 100 words)
        "emoji": len(EMOJI_RE.findall(text)) * p,
        "exclaim": text.count("!") * p,
        "questions": text.count("?") * p,
        "em_dash": text.count("—") * p,
        # ends with a follow-up question
        "ends_with_question": float(text.rstrip().endswith("?")),
    }

feats = pd.DataFrame([features(t) for t in data["llm_output"]])
feat = pd.concat([data[["family", "model", "split"]].reset_index(drop=True), feats], axis=1)

pd.set_option("display.float_format", "{:.2f}".format)
cols = [c for c in feats.columns]

print("Mean of each feature per family ")
print(feat.groupby("family")[cols].mean().T)

print("\nMedian length per family ")
print(feat.groupby("family")[["chars", "words", "lines", "paragraphs"]].median().T)

fmt = ["headers", "bold", "bullets", "numbered", "table_rows", "hrules", "code_blocks", "emoji", "em_dash"]
print("\n=== % of responses that use each formatting feature at least once ===")
print((feat[fmt] > 0).groupby(feat["family"]).mean().mul(100).T)

block 16

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

groups = {
    "length":      ["chars", "words", "lines", "paragraphs"],
    "formatting":  ["headers", "bold", "bullets", "numbered", "table_rows", "hrules", "code_blocks"],
    "punctuation": ["emoji", "exclaim", "questions", "em_dash", "ends_with_question"],
    "lexical":     ["avg_word_len", "words_per_sentence", "mattr"],
}
all_feats = [c for g in groups.values() for c in g]
fams = ["Claude", "GPT", "Gemini"]

# Heavy-tailed counts are log-transformed so a few huge responses don't dominate
raw_cols = ["mattr", "avg_word_len", "ends_with_question"]
Xt = feat[all_feats].copy()
for c in all_feats:
    if c not in raw_cols:
        Xt[c] = np.log1p(Xt[c])

#Effect sizes
rows = []
for c in all_feats:
    s = pd.DataFrame({"family": feat["family"], "v": Xt[c]}).dropna()
    grand = s["v"].mean()
    ss_tot = ((s["v"] - grand) ** 2).sum()
    ss_between = sum(len(g) * (g["v"].mean() - grand) ** 2 for _, g in s.groupby("family"))
    row = {"feature": c, "eta_sq": ss_between / ss_tot}
    for a, b in combinations(fams, 2):
        x, y_ = s.loc[s.family == a, "v"], s.loc[s.family == b, "v"]
        sp = np.sqrt(((len(x) - 1) * x.var() + (len(y_) - 1) * y_.var()) / (len(x) + len(y_) - 2))
        row[f"d: {a} - {b}"] = (x.mean() - y_.mean()) / sp if sp > 0 else 0.0
    rows.append(row)

eff = pd.DataFrame(rows).set_index("feature").sort_values("eta_sq", ascending=False)
print("=== Effect sizes (sorted by eta squared) ===")
print("eta_sq: share of variance explained by family (0.01 small, 0.06 medium, 0.14 large)")
print("d: Cohen's d; positive means the FIRST family is higher (0.2 small, 0.5 medium, 0.8 large)\n")
print(eff.round(2))

#  Features-only logistic regression with ablations
tr = (feat["split"] == "train").values
te = (feat["split"] == "test").values
y = feat["family"].values

Xt = Xt.fillna(Xt[tr].median())      # only mattr has missing values (responses under 100 words)

experiments = {
    "length only":          groups["length"],
    "formatting only":      groups["formatting"],
    "punctuation only":     groups["punctuation"],
    "lexical/sentence only": groups["lexical"],
    "all features":         all_feats,
    "all minus length":     [c for c in all_feats if c not in groups["length"]],
}

res = []
for name, cols in experiments.items():
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Xt.loc[tr, cols], y[tr])
    p = clf.predict(Xt.loc[te, cols])
    res.append({"features": name, "n_features": len(cols),
                "test_acc": accuracy_score(y[te], p),
                "macro_f1": f1_score(y[te], p, average="macro")})

print("\n=== Features-only logistic regression (test set) ===")
print(pd.DataFrame(res).set_index("features").round(3))
print("chance:", round(1 / len(fams), 3))

RQ2 block 17

In [ ]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

# Add a separator token so the model can tell where the prompt ends
if "<sep>" not in stoi:
    itos.append("<sep>"); stoi["<sep>"] = len(itos) - 1
VOCAB = len(itos)
SEP = stoi["<sep>"]

IN_LEN = 128                 # prompt tokens kept the output window stays at max (512)

def make_input_loaders(df, max_len=MAX_LEN):
    """[prompt tokens] only, first max_len tokens, padded, with true lengths."""
    out = {}
    for s in ["train", "val", "test"]:
        g = df[df["split"] == s]
        X_rows, L_rows = [], []
        for text in g["llm_input"]:
            ids = [stoi.get(t, UNK) for t in tokenize(text)[:max_len]]
            L_rows.append(max(len(ids), 1))
            X_rows.append(ids + [PAD] * (max_len - len(ids)))
        X = torch.tensor(X_rows, dtype=torch.long)
        L = torch.tensor(L_rows, dtype=torch.long)
        y = torch.tensor(g["family"].map(label2id).values, dtype=torch.long)
        out[s] = DataLoader(TensorDataset(X, L, y), batch_size=BATCH, shuffle=(s == "train"))
    return out

def make_pair_loaders(df, in_len=IN_LEN, out_len=MAX_LEN):
    """[prompt tokens] <sep> [response tokens], as one sequence."""
    total = in_len + 1 + out_len
    def enc(x, y):
        a = [stoi.get(t, UNK) for t in tokenize(x)[:in_len]]
        b = [stoi.get(t, UNK) for t in tokenize(y)[:out_len]]
        ids = a + [SEP] + b
        n = len(ids)
        return ids + [PAD] * (total - n), n
    out = {}
    for s in ["train", "val", "test"]:
        g = df[df["split"] == s]
        e = [enc(x, y) for x, y in zip(g["llm_input"], g["llm_output"])]
        X = torch.tensor([r[0] for r in e], dtype=torch.long)
        L = torch.tensor([r[1] for r in e], dtype=torch.long)
        y = torch.tensor(g["family"].map(label2id).values, dtype=torch.long)
        out[s] = DataLoader(TensorDataset(X, L, y), batch_size=BATCH, shuffle=(s == "train"))
    return out

builders = {
    "Input only":     lambda: make_input_loaders(data),
    "Input + output": lambda: make_pair_loaders(data),
}

rq2 = {}
for setting, build in builders.items():
    ldrs = build()
    print(f"\n {setting} ")
    print("train/val/test sizes:", {s: len(l.dataset) for s, l in ldrs.items()})
    for mname, cls in [("CNN", CNN), ("BiLSTM", BiLSTM)]:
        print(f"\n--- {mname} ---")
        set_seed(42)
        model, best_val = train_model(cls(), ldrs)
        r = evaluate(model, ldrs["test"], f"{mname} | {setting}")
        r["val_acc"] = best_val
        rq2[(setting, mname)] = r
    del ldrs

#  Summary
for m in ["CNN", "BiLSTM"]:
    rq2[("Output only", m)] = {"acc": results[m]["acc"], "macro_f1": results[m]["macro_f1"]}

rows = []
for setting in ["Input only", "Output only", "Input + output"]:
    row = {"setting": setting}
    for m in ["CNN", "BiLSTM"]:
        row[f"{m} acc"] = rq2[(setting, m)]["acc"]
        row[f"{m} F1"] = rq2[(setting, m)]["macro_f1"]
    rows.append(row)

print("\n RQ2 summary (test set) ")
print(pd.DataFrame(rows).set_index("setting").round(4))
print("chance:", round(1 / N_CLASSES, 4))